# Baseline LSTM v3 — Ranking + Confidence Revision

**Author:** Minho  
**Project:** Risk-Adjusted Portfolio Optimization  
**Model direction:** LSTM with cross-sectional ranking signal and confidence-aware portfolio sizing  
**Dataset:** configurable through `DATASET_NAME` / `dataset`; designed to transfer to the hidden biotech-style universe  
**MLflow run name:** `Minho_RankingConfidence_LSTM_v3`

---

### Revision goal

The previous LSTM was stable because it held a broad book, but the hidden biotech review showed that the portfolio was too diluted: average active names were about 133, average max weight was about 1.49%, and effective names were about 100. That diversification helped drawdown control, but it muted exposure to large event-driven winners such as SMMT, IVVD, SRRK, PRAX, SLNO, REPL, KOD, and OLMA.

This version keeps the broad LSTM base, then adds ranking and confidence machinery so concentration becomes a deliberate choice instead of an accident. The model should stay diversified when the cross-sectional signal is weak or the market regime is fragile, and size up a smaller sleeve when the opportunity spread is unusually strong.


### Meeting-note guardrails

This notebook treats December 31, 2022 as the final model-selection cutoff. Training and validation can use information up to that date, but the main reported test period begins after the cutoff so post-2022 performance is a real holdout. The portfolio layer also uses softmax sizing, explicit volatility/downside penalties, active-name floors, max-weight caps, and stress diagnostics that try to find how the strategy loses money.

### What this notebook now targets

1. Build shared technical features plus cross-sectional, market-regime, breadth, dispersion, drawdown, and volume-shock features.
2. Train a sequence model with pointwise alpha plus a ranking/top-quintile component.
3. Add auxiliary heads for confidence, volatility, and downside risk.
4. Emit predictions with `expected_return`, `confidence`, and risk columns.
5. Compare broad allocation against top 20, top 40, top 60, top 100, and dynamic-confidence sleeves.
6. Track performance by regime, benchmark stress days, active names, effective names, max weight, and turnover.
7. Log the final selected portfolio and diagnostics to MLflow.

### How to run

```
cd <repo-root>
source venv312/bin/activate
jupyter notebook baseline_lstm.ipynb
```

Run all cells top to bottom after choosing the experiment settings in the config cell.


## 0. Bootstrap — locate repo root

In [1]:
import os
import sys
from pathlib import Path

# --- If auto-detection fails, set this manually ---
# repo_root = Path('/Users/minhochoi/Portfolio-Optimization-Lib') 
# --------------------------------------------------

def _is_repo_root(p: Path) -> bool:
    return (p / 'pyproject.toml').exists() and (p / 'src' / 'portfolio_toolkit').exists()

if 'repo_root' not in dir() or not _is_repo_root(Path(repo_root)):
    _candidates = [Path.cwd(), *Path.cwd().parents]
    _found = next((p for p in _candidates if _is_repo_root(p)), None)
    if _found is None:
        raise RuntimeError(
            'Cannot find repo root. Uncomment and set repo_root manually at the top of this cell.'
        )
    repo_root = _found

repo_root = Path(repo_root).resolve()
os.chdir(repo_root)

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('python:   ', sys.executable)

repo_root: /Users/minhochoi/Portfolio-Optimization-Lib
python:    /Users/minhochoi/Portfolio-Optimization-Lib/venv312/bin/python


## 1. Imports

In [2]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    custom_dataset,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

/Users/minhochoi/Portfolio-Optimization-Lib/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0 | CUDA: False


--------------------------------------------------------------------------------

### `build_model_features(prices)` & `predict_from_prices(model, prices, dates=None, tickers=None)`

In [3]:
# Reusable model functions

def _add_market_regime_features(prices: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    """Attach date-level benchmark, breadth, dispersion, and regime context."""
    panel = prices.sort_values(['ticker', 'date']).copy()
    panel['ret_1d'] = panel.groupby('ticker')['adj_close'].pct_change()
    panel['ret_5d'] = panel.groupby('ticker')['adj_close'].pct_change(5)
    panel['ret_20d'] = panel.groupby('ticker')['adj_close'].pct_change(20)

    spy = panel.loc[panel['ticker'] == 'SPY', ['date', 'adj_close', 'ret_1d', 'ret_5d', 'ret_20d']].copy()
    spy = spy.sort_values('date')
    spy['spy_vol_20d'] = spy['ret_1d'].rolling(20, min_periods=20).std(ddof=0)
    spy['spy_vol_60d'] = spy['ret_1d'].rolling(60, min_periods=60).std(ddof=0)
    spy['spy_sma_20d'] = spy['adj_close'].rolling(20, min_periods=20).mean()
    spy['spy_sma_60d'] = spy['adj_close'].rolling(60, min_periods=60).mean()
    spy['spy_trend_20d'] = spy['adj_close'] / spy['spy_sma_20d'] - 1.0
    spy['spy_trend_60d'] = spy['adj_close'] / spy['spy_sma_60d'] - 1.0
    spy['spy_drawdown_60d'] = spy['adj_close'] / spy['adj_close'].rolling(60, min_periods=20).max() - 1.0
    spy['spy_rebound_5d'] = spy['ret_5d'].clip(lower=0.0)

    tradable_panel = panel.loc[panel['ticker'] != 'SPY'].copy()
    breadth = tradable_panel.groupby('date').agg(
        universe_breadth_20d=('ret_20d', lambda s: float((s > 0).mean())),
        universe_dispersion_5d=('ret_5d', 'std'),
        universe_dispersion_20d=('ret_20d', 'std'),
        universe_median_return_5d=('ret_5d', 'median'),
        universe_median_return_20d=('ret_20d', 'median'),
    ).reset_index()

    regime = spy[[
        'date', 'ret_5d', 'ret_20d', 'spy_vol_20d', 'spy_vol_60d',
        'spy_trend_20d', 'spy_trend_60d', 'spy_drawdown_60d', 'spy_rebound_5d'
    ]].rename(columns={'ret_5d': 'spy_return_5d', 'ret_20d': 'spy_return_20d'})
    regime = regime.merge(breadth, on='date', how='left')
    regime['high_vol_regime'] = (regime['spy_vol_20d'] >= regime['spy_vol_20d'].rolling(252, min_periods=60).median()).astype(float)
    regime['drawdown_regime'] = (regime['spy_drawdown_60d'] <= -0.08).astype(float)
    regime['rebound_regime'] = ((regime['spy_return_5d'] > 0.03) & (regime['spy_drawdown_60d'] > -0.12)).astype(float)

    return frame.merge(regime, on='date', how='left')


def _add_cross_sectional_features(frame: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Add within-date ranks and z-scores so the LSTM sees relative opportunity."""
    out = frame.copy()
    grouped = out.groupby('date', sort=False)
    for col in base_cols:
        if col not in out.columns:
            continue
        out[f'{col}_cs_rank'] = grouped[col].rank(pct=True, method='average')
        mean = grouped[col].transform('mean')
        std = grouped[col].transform('std').replace(0.0, np.nan)
        out[f'{col}_cs_z'] = (out[col] - mean) / std
    return out


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Build the full feature matrix from raw prices.

    v3 adds cross-sectional ranks/z-scores and market-regime context so the
    sequence model can distinguish ordinary momentum from unusually strong
    relative opportunity in calm, drawdown, and rebound regimes.
    """
    base_features = [
        'return_1d', 'return_5d', 'return_10d', 'return_20d', 'return_60d',
        'vol_5d', 'vol_10d', 'vol_20d', 'vol_60d',
        'downside_vol_20d', 'upside_vol_20d', 'atr_14',
        'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_60d', 'momentum_120d',
        'rsi_14', 'macd_hist', 'bollinger_z_20d',
        'beta_20d_spy', 'beta_60d_spy',
        'excess_return_5d_vs_spy', 'excess_return_20d_vs_spy', 'excess_return_60d_vs_spy',
        'relative_momentum_20d_vs_spy',
        'volume_zscore_20d', 'volume_zscore_60d', 'dollar_volume_ratio_20d',
        'price_to_sma_20d', 'price_to_sma_50d', 'price_to_sma_200d',
        'intraday_range', 'close_open_gap', 'close_location_in_range',
        'distance_to_20d_high', 'distance_to_20d_low',
        'distance_to_60d_high', 'distance_to_60d_low',
        'skew_20d', 'kurtosis_20d',
    ]

    feature_frame = build_features(prices, feature_names=base_features)

    panel = prices.sort_values(['ticker', 'date']).copy()
    ret5  = panel.groupby('ticker')['adj_close'].pct_change(5)
    ret20 = panel.groupby('ticker')['adj_close'].pct_change(20)
    ret60 = panel.groupby('ticker')['adj_close'].pct_change(60)

    custom_df = panel[['date', 'ticker']].copy()
    custom_df['price_accel'] = (ret5 - ret20).values
    custom_df['momentum_reversal_20_60'] = (ret20 - ret60).values
    custom_df['vol_compression_20_60'] = (
        panel.groupby('ticker')['adj_close'].pct_change()
        .groupby(panel['ticker']).transform(lambda s: s.rolling(20, min_periods=20).std(ddof=0))
        /
        panel.groupby('ticker')['adj_close'].pct_change()
        .groupby(panel['ticker']).transform(lambda s: s.rolling(60, min_periods=60).std(ddof=0)).replace(0.0, np.nan)
    ).values

    frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
    frame = _add_market_regime_features(prices, frame)

    cs_cols = [
        'momentum_20d', 'momentum_60d', 'excess_return_20d_vs_spy',
        'relative_momentum_20d_vs_spy', 'vol_20d', 'downside_vol_20d',
        'volume_zscore_20d', 'price_accel', 'distance_to_20d_high',
    ]
    frame = _add_cross_sectional_features(frame, cs_cols)
    return frame


def _model_outputs(model: nn.Module, X_tensor: torch.Tensor):
    outputs = model(X_tensor)
    if isinstance(outputs, dict):
        return outputs
    return {'alpha': outputs}


def predict_from_prices(
    model: nn.Module,
    prices: pd.DataFrame,
    train_mean: pd.Series,
    train_std: pd.Series,
    feature_names: list,
    seq_len: int,
    horizon: int,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    """Generate the standard prediction frame from raw prices."""
    frame = build_model_features(prices)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_names)

    X_scaled = ((frame[feature_names] - train_mean) / train_std).to_numpy(dtype=np.float32)

    X_seqs, meta = [], []
    frame = frame.reset_index(drop=True)
    for ticker, grp in frame.groupby('ticker', sort=False):
        idx = grp.index.tolist()
        dates_grp = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            X_seqs.append(X_scaled[rows])
            meta.append((dates_grp[end], ticker))

    if not X_seqs:
        raise ValueError('No sequences could be built; not enough rows per ticker.')

    model.eval()
    device = next(model.parameters()).device
    alpha_list, confidence_list, vol_list, downside_list = [], [], [], []
    with torch.no_grad():
        for i in range(0, len(X_seqs), 2048):
            X_tensor = torch.as_tensor(np.stack(X_seqs[i:i+2048]), dtype=torch.float32, device=device)
            outputs = _model_outputs(model, X_tensor)
            alpha_list.append(outputs['alpha'].detach().cpu().numpy())
            if 'confidence_logit' in outputs:
                confidence_list.append(torch.sigmoid(outputs['confidence_logit']).detach().cpu().numpy())
            if 'volatility' in outputs:
                vol_list.append(torch.nn.functional.softplus(outputs['volatility']).detach().cpu().numpy())
            if 'downside' in outputs:
                downside_list.append(torch.nn.functional.softplus(outputs['downside']).detach().cpu().numpy())

    pred_dates, pred_tickers = zip(*meta)
    predictions = pd.DataFrame({
        'date': pd.to_datetime(list(pred_dates)),
        'ticker': list(pred_tickers),
        'horizon': horizon,
        'expected_return': np.concatenate(alpha_list).astype(float),
    })
    predictions['expected_alpha'] = predictions['expected_return']
    if confidence_list:
        predictions['confidence'] = np.concatenate(confidence_list).astype(float)
        predictions['uncertainty'] = 1.0 - predictions['confidence']
    else:
        predictions['confidence'] = 0.5
        predictions['uncertainty'] = 0.5
    if vol_list:
        predictions['expected_volatility'] = np.concatenate(vol_list).astype(float)
    if downside_list:
        predictions['expected_downside'] = np.concatenate(downside_list).astype(float)

    predictions = predictions.drop_duplicates(['date', 'ticker', 'horizon'], keep='last')

    if tickers is not None:
        predictions = predictions[predictions['ticker'].isin(tickers)]
    if dates is not None:
        predictions = predictions[predictions['date'].isin(pd.to_datetime(dates))]

    return predictions.sort_values(['date', 'ticker']).reset_index(drop=True)


print('v3 feature builder and prediction helper defined.')


build_model_features() and predict_from_prices() defined.


----------------------------------------------------------------------------------

## 2. Config

The first revision pass keeps the original LSTM backbone runnable, but promotes the new experiment knobs into the config: sequence-length sweep, ranking/classification loss weights, confidence-based concentration, sleeve sizes, rebalance frequencies, and portfolio risk controls.


### Retired baseline config

The original `shared_set_1` config has been retired from the executable path. The active config below is the v3 ranking/confidence setup. Keeping this note avoids accidentally running the stale baseline cell with the invalid device branch and old run name.


### Active v3 experiment config

Use this cell to choose the local training universe and the ranking/confidence portfolio behavior. The hidden biotech review is treated as the target deployment context: broad by default, concentrated only when the model signal is strong.


In [6]:
from portfolio_toolkit import custom_dataset
from dataclasses import replace
from datetime import date

TRAINING_CUTOFF = date(2022, 12, 31)

# Local development universe. Hidden evaluation can replace this dataset while reusing
# the same feature, model, prediction, and portfolio code.
SUBSET_TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AVGO', 'ORCL', 'ADBE', 'CRM', 'AMD', 'INTC',
    # Financials
    'JPM', 'BAC', 'GS', 'MS', 'BLK', 'SPGI', 'V', 'MA',
    # Healthcare / biotech-adjacent large caps
    'LLY', 'UNH', 'JNJ', 'ABBV', 'MRK', 'TMO',
    # Consumer
    'AMZN', 'TSLA', 'HD', 'MCD', 'NKE', 'SBUX',
    # Industrials
    'CAT', 'DE', 'UPS', 'HON', 'GE', 'RTX',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB',
    # Communication
    'DIS', 'NFLX', 'CMCSA', 'T',
    # Utilities / defensives
    'PG', 'KO', 'PEP', 'WMT', 'COST',
]

dataset = custom_dataset(
    tickers   = SUBSET_TICKERS,
    start     = '2014-01-02',
    end       = '2025-12-31',
    benchmark = 'SPY',
    name      = 'sp500_subset_50_v3_dev',
)

dataset = replace(
    dataset,
    train_start = date(2014, 1,  2),
    train_end   = date(2021, 12, 31),
    val_start   = date(2022, 1,  3),
    val_end     = TRAINING_CUTOFF,
    test_start  = date(2023, 1,  3),
    test_end    = date(2025, 12, 31),
)

# Core experiment identity
DATASET_NAME     = dataset
RUN_NAME         = 'Minho_RankingConfidence_LSTM_v3'
MODEL_FAMILY     = 'ranking_confidence_lstm'
TARGET_TYPE      = 'alpha_rank_confidence'

# Forecasting setup
HORIZON          = 5
SEQ_LEN          = 20
SEQ_LEN_SWEEP    = [10, 20, 40, 60]
MODEL_SWEEP      = ['lstm']  # TODO: add gru, temporal_cnn, mlp after LSTM v3 is stable

# Model hyperparameters
HIDDEN_SIZE      = 64
NUM_LAYERS       = 2
DROPOUT          = 0.2
LR               = 1e-3
EPOCHS           = 30
BATCH_SIZE       = 512
SEED             = 42

# Multi-task loss weights
ALPHA_LOSS_WEIGHT      = 1.00
RANK_LOSS_WEIGHT       = 0.35
TOP_QUINTILE_WEIGHT    = 0.25
VOL_LOSS_WEIGHT        = 0.15
DOWNSIDE_LOSS_WEIGHT   = 0.20

# Portfolio experiment knobs
REBALANCE_FREQ         = 5
REBALANCE_FREQ_SWEEP   = [5, 10, 21]
SLEEVE_SIZES           = [20, 40, 60, 100]
BROAD_TOP_FRACTION     = 0.55
CONFIDENCE_TOP_FRACTION= 0.35
CONFIDENCE_THRESHOLD   = 0.62
SOFTMAX_TEMPERATURE    = 0.35
RISK_AVERSION          = 1.25
DOWNSIDE_AVERSION      = 1.50
MAX_WEIGHT             = 0.025
MIN_ACTIVE_NAMES       = 50
MIN_EFFECTIVE_NAMES    = 25
NO_TRADE_BAND          = 0.0025
TARGET_BETA            = None

artifact_dir     = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec             = dataset
BENCHMARK_TICKER = spec.benchmark_ticker
TRADABLE_TICKERS = [t for t in spec.tickers if t != BENCHMARK_TICKER]


assert spec.train_end <= TRAINING_CUTOFF, 'Train split leaks beyond the Dec 2022 cutoff'
assert spec.val_end <= TRAINING_CUTOFF, 'Validation/model-selection split leaks beyond the Dec 2022 cutoff'
assert spec.test_start > TRAINING_CUTOFF, 'Main test split must start after Dec 2022'

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Dataset  : {dataset.name}  ({len(TRADABLE_TICKERS)} tradable tickers)')
print(f'Benchmark: {BENCHMARK_TICKER}')
print(f'Train    : {spec.train_start} -> {spec.train_end}')
print(f'Val      : {spec.val_start}   -> {spec.val_end}')
print(f'Test     : {spec.test_start}  -> {spec.test_end}')
print(f'Device   : {DEVICE}')
print(f'Run name : {RUN_NAME}')
print(f'Cutoff   : no training/model selection beyond {TRAINING_CUTOFF}')


Dataset  : sp500_subset_50  (50 tradable tickers)
Benchmark: SPY
Train    : 2014-01-02 → 2020-12-31
Val      : 2021-01-04   → 2021-12-31
Test     : 2022-01-03  → 2025-12-31
Device   : mps


## 3. Load Prices

In [7]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print('Shape     :', prices.shape)
print('Date range:', prices['date'].min().date(), '→', prices['date'].max().date())
prices.head(3)

Shape     : (150900, 8)
Date range: 2014-01-02 → 2025-12-31


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140661,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764154,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855570,412610800


## 4. Features — Shared Toolkit + v3 Ranking/Regime Features

The baseline used 16 shared features plus `price_accel`. v3 keeps those signals and adds features aligned with the task review: cross-sectional ranks/z-scores, benchmark trend and drawdown state, universe breadth and dispersion, volume shock, volatility compression, and relative strength.


In [8]:
frame = build_model_features(prices)

BASE_FEATURES = [
    'return_1d', 'return_5d', 'return_10d', 'return_20d', 'return_60d',
    'vol_5d', 'vol_10d', 'vol_20d', 'vol_60d',
    'downside_vol_20d', 'upside_vol_20d', 'atr_14',
    'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_60d', 'momentum_120d',
    'rsi_14', 'macd_hist', 'bollinger_z_20d',
    'beta_20d_spy', 'beta_60d_spy',
    'excess_return_5d_vs_spy', 'excess_return_20d_vs_spy', 'excess_return_60d_vs_spy',
    'relative_momentum_20d_vs_spy',
    'volume_zscore_20d', 'volume_zscore_60d', 'dollar_volume_ratio_20d',
    'price_to_sma_20d', 'price_to_sma_50d', 'price_to_sma_200d',
    'intraday_range', 'close_open_gap', 'close_location_in_range',
    'distance_to_20d_high', 'distance_to_20d_low', 'distance_to_60d_high', 'distance_to_60d_low',
    'skew_20d', 'kurtosis_20d',
]

CUSTOM_FEATURES = [
    'price_accel', 'momentum_reversal_20_60', 'vol_compression_20_60',
]

REGIME_FEATURES = [
    'spy_return_5d', 'spy_return_20d', 'spy_vol_20d', 'spy_vol_60d',
    'spy_trend_20d', 'spy_trend_60d', 'spy_drawdown_60d', 'spy_rebound_5d',
    'universe_breadth_20d', 'universe_dispersion_5d', 'universe_dispersion_20d',
    'universe_median_return_5d', 'universe_median_return_20d',
    'high_vol_regime', 'drawdown_regime', 'rebound_regime',
]

CS_FEATURES = [
    c for c in frame.columns
    if c.endswith('_cs_rank') or c.endswith('_cs_z')
]

ALL_FEATURES = BASE_FEATURES + CUSTOM_FEATURES + REGIME_FEATURES + CS_FEATURES
missing_features = sorted(set(ALL_FEATURES) - set(frame.columns))
assert not missing_features, f'Missing features: {missing_features}'

print(f'Base features      : {len(BASE_FEATURES)}')
print(f'Custom features    : {len(CUSTOM_FEATURES)}')
print(f'Regime features    : {len(REGIME_FEATURES)}')
print(f'Cross-sec features : {len(CS_FEATURES)}')
print(f'Total features     : {len(ALL_FEATURES)}')

frame[['date', 'ticker', 'price_accel', 'spy_drawdown_60d', 'universe_dispersion_20d']].dropna().head(3)


Features: 17  (16 shared + 1 custom: price_accel)


,date,ticker,price_accel
20,2014-01-31,AAPL,0.011701
21,2014-02-03,AAPL,-0.016032
22,2014-02-04,AAPL,0.069125


## 5. Targets + Train / Val / Test Split

v3 keeps the 5-day SPY-relative alpha target, then adds auxiliary labels for cross-sectional ranking, top-quintile membership, realized volatility, and downside risk. The ranking/classification labels are date-local so the model learns which names are attractive relative to the same day's opportunity set.


In [9]:
TARGET_COL = f'forward_alpha_{HORIZON}d_vs_spy'
FWD_RETURN_COL = f'forward_return_{HORIZON}d'
VOL_TARGET_COL = f'forward_realized_vol_{HORIZON}d'
DOWNSIDE_TARGET_COL = f'forward_downside_vol_{HORIZON}d'
TARGET_RANK_COL = 'target_alpha_cs_rank'
TOP_QUINTILE_COL = 'target_top_quintile'

alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)

panel = prices.sort_values(['ticker', 'date']).copy()
panel[FWD_RETURN_COL] = panel.groupby('ticker')['adj_close'].shift(-HORIZON) / panel['adj_close'] - 1.0

future_returns = []
for step in range(1, HORIZON + 1):
    future_returns.append(panel.groupby('ticker')['adj_close'].shift(-step) / panel.groupby('ticker')['adj_close'].shift(-(step - 1)) - 1.0)
future_return_frame = pd.concat(future_returns, axis=1)
panel[VOL_TARGET_COL] = future_return_frame.std(axis=1, ddof=0) * np.sqrt(252.0)
panel[DOWNSIDE_TARGET_COL] = future_return_frame.clip(upper=0.0).std(axis=1, ddof=0) * np.sqrt(252.0)

risk_targets = panel[['date', 'ticker', FWD_RETURN_COL, VOL_TARGET_COL, DOWNSIDE_TARGET_COL]]

target_frame = (
    frame
    .merge(alpha_targets, on=['date', 'ticker'], how='left')
    .merge(risk_targets, on=['date', 'ticker'], how='left')
)
target_frame = target_frame[target_frame['ticker'].isin(TRADABLE_TICKERS)].copy()

target_frame[TARGET_RANK_COL] = target_frame.groupby('date')[TARGET_COL].rank(pct=True, method='average')
target_frame[TOP_QUINTILE_COL] = (target_frame[TARGET_RANK_COL] > 0.80).astype(float)
target_frame['target_alpha_cs_z'] = (
    target_frame[TARGET_COL] - target_frame.groupby('date')[TARGET_COL].transform('mean')
) / target_frame.groupby('date')[TARGET_COL].transform('std').replace(0.0, np.nan)

target_cols = [TARGET_COL, TARGET_RANK_COL, TOP_QUINTILE_COL, VOL_TARGET_COL, DOWNSIDE_TARGET_COL]
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=ALL_FEATURES + target_cols).reset_index(drop=True)

train = slice_split(target_frame, DATASET_NAME, 'train', repo_root=repo_root)
val   = slice_split(target_frame, DATASET_NAME, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, DATASET_NAME, 'test',  repo_root=repo_root)

train_mean = train[ALL_FEATURES].mean()
train_std  = train[ALL_FEATURES].std(ddof=0).replace(0.0, 1.0)

def scale(df):
    return ((df[ALL_FEATURES] - train_mean) / train_std).to_numpy(dtype=np.float32)

X_train, X_val, X_test = scale(train), scale(val), scale(test)

print(f'Train : {len(train):>6,} rows    X_train : {X_train.shape}')
print(f'Val   : {len(val):>6,} rows    X_val   : {X_val.shape}')
print(f'Test  : {len(test):>6,} rows    X_test  : {X_test.shape}')
print(f'Top-quintile train rate: {train[TOP_QUINTILE_COL].mean():.2%}')


Train : 85,148 rows    X_train : (85148, 17)
Val   : 12,600 rows    X_val   : (12600, 17)
Test  : 49,892 rows    X_test  : (49892, 17)


## 6. Build Per-Ticker Sequences

Sequences still stay inside ticker boundaries. The label bundle is taken from the last timestep of each sequence and now includes alpha, cross-sectional rank, top-quintile class, volatility, and downside risk.


In [10]:
def make_sequences(
    df: pd.DataFrame,
    X: np.ndarray,
    target_columns: list[str],
    seq_len: int,
):
    """Return (X_seq, y_dict, meta) where meta is list of (date, ticker)."""
    Xs, ys, meta = [], {col: [] for col in target_columns}, []
    df = df.reset_index(drop=True)
    for ticker, grp in df.groupby('ticker', sort=False):
        idx = grp.index.tolist()
        dates = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            row_idx = idx[end]
            Xs.append(X[rows])
            for col in target_columns:
                ys[col].append(df.loc[row_idx, col])
            meta.append((dates[end], ticker))

    y_dict = {col: np.asarray(vals, dtype=np.float32) for col, vals in ys.items()}
    return np.stack(Xs).astype(np.float32), y_dict, meta

TARGET_COLUMNS = [TARGET_COL, TARGET_RANK_COL, TOP_QUINTILE_COL, VOL_TARGET_COL, DOWNSIDE_TARGET_COL]

print('Building sequences ...')
X_tr_seq, y_tr_seq, meta_train = make_sequences(train, X_train, TARGET_COLUMNS, SEQ_LEN)
X_va_seq, y_va_seq, meta_val   = make_sequences(val,   X_val,   TARGET_COLUMNS, SEQ_LEN)
X_te_seq, y_te_seq, meta_test  = make_sequences(test,  X_test,  TARGET_COLUMNS, SEQ_LEN)

y_train = y_tr_seq[TARGET_COL]
y_val   = y_va_seq[TARGET_COL]

print(f'Train sequences : {X_tr_seq.shape}')
print(f'Val sequences   : {X_va_seq.shape}')
print(f'Test sequences  : {X_te_seq.shape}')
print(f'Train top-quintile sequences: {y_tr_seq[TOP_QUINTILE_COL].mean():.2%}')


Building sequences ...
Train sequences : (84198, 20, 17)
Val sequences   : (11650, 20, 17)
Test sequences  : (48942, 20, 17)


## 7. Ranking Confidence LSTM

The baseline LSTM predicted one scalar alpha with MSE. v3 keeps that alpha head but adds a confidence/top-quintile head plus volatility and downside-risk heads. Training combines pointwise alpha error, pairwise ranking pressure, top-quintile classification, and risk-head losses.


In [11]:
class RankingConfidenceLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_head = nn.Linear(hidden_size, 1)
        self.confidence_head = nn.Linear(hidden_size, 1)
        self.volatility_head = nn.Linear(hidden_size, 1)
        self.downside_head = nn.Linear(hidden_size, 1)
        for head in [self.alpha_head, self.confidence_head, self.volatility_head, self.downside_head]:
            nn.init.xavier_uniform_(head.weight)
            nn.init.zeros_(head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(self.norm(h_n[-1]))
        return {
            'alpha': self.alpha_head(h).squeeze(-1),
            'confidence_logit': self.confidence_head(h).squeeze(-1),
            'volatility': self.volatility_head(h).squeeze(-1),
            'downside': self.downside_head(h).squeeze(-1),
        }


# Backward-compatible alias for checkpoint reproduction cells.
BaselineLSTM = RankingConfidenceLSTM


def _as_tensor_dict(y_dict, start, stop, device):
    return {
        key: torch.as_tensor(val[start:stop], dtype=torch.float32, device=device)
        for key, val in y_dict.items()
    }


def _pairwise_rank_loss(pred, actual, max_pairs=4096):
    n = pred.shape[0]
    if n < 2:
        return pred.new_tensor(0.0)
    pair_count = min(max_pairs, n * 4)
    i = torch.randint(0, n, (pair_count,), device=pred.device)
    j = torch.randint(0, n, (pair_count,), device=pred.device)
    direction = torch.sign(actual[i] - actual[j])
    mask = direction != 0
    if not mask.any():
        return pred.new_tensor(0.0)
    margin = (pred[i] - pred[j]) * direction
    return torch.nn.functional.softplus(-margin[mask]).mean()


def _loss_from_outputs(outputs, yb):
    alpha_loss = torch.nn.functional.mse_loss(outputs['alpha'], yb[TARGET_COL])
    rank_loss = _pairwise_rank_loss(outputs['alpha'], yb[TARGET_COL])
    top_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs['confidence_logit'], yb[TOP_QUINTILE_COL]
    )
    vol_loss = torch.nn.functional.mse_loss(
        torch.nn.functional.softplus(outputs['volatility']), yb[VOL_TARGET_COL]
    )
    downside_loss = torch.nn.functional.mse_loss(
        torch.nn.functional.softplus(outputs['downside']), yb[DOWNSIDE_TARGET_COL]
    )
    total = (
        ALPHA_LOSS_WEIGHT * alpha_loss
        + RANK_LOSS_WEIGHT * rank_loss
        + TOP_QUINTILE_WEIGHT * top_loss
        + VOL_LOSS_WEIGHT * vol_loss
        + DOWNSIDE_LOSS_WEIGHT * downside_loss
    )
    return total, {
        'alpha_loss': float(alpha_loss.detach().cpu()),
        'rank_loss': float(rank_loss.detach().cpu()),
        'top_loss': float(top_loss.detach().cpu()),
        'vol_loss': float(vol_loss.detach().cpu()),
        'downside_loss': float(downside_loss.detach().cpu()),
    }


def _predict_batched(model, X, device, batch_size=2048):
    model.eval()
    parts = {'alpha': [], 'confidence': [], 'volatility': [], 'downside': []}
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            Xb = torch.as_tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
            out = _model_outputs(model, Xb)
            parts['alpha'].append(out['alpha'].detach().cpu().numpy())
            parts['confidence'].append(torch.sigmoid(out['confidence_logit']).detach().cpu().numpy())
            parts['volatility'].append(torch.nn.functional.softplus(out['volatility']).detach().cpu().numpy())
            parts['downside'].append(torch.nn.functional.softplus(out['downside']).detach().cpu().numpy())
    return {key: np.concatenate(vals) for key, vals in parts.items()}


def _eval_loss_batched(model, X, y_dict, device, batch_size=2048):
    total, n = 0.0, 0
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            stop = min(i + batch_size, len(X))
            Xb = torch.as_tensor(X[i:stop], dtype=torch.float32, device=device)
            yb = _as_tensor_dict(y_dict, i, stop, device)
            loss, _ = _loss_from_outputs(_model_outputs(model, Xb), yb)
            total += float(loss) * (stop - i)
            n += stop - i
    return total / max(n, 1)


def train_lstm(model, X_tr, y_tr, X_va, y_va, epochs, batch_size, lr, device):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    order = np.arange(len(X_tr))
    history = []

    for epoch in range(epochs):
        model.train()
        np.random.shuffle(order)
        batch_losses = []
        for start in range(0, len(order), batch_size):
            batch_idx = order[start:start + batch_size]
            Xb = torch.as_tensor(X_tr[batch_idx], dtype=torch.float32, device=device)
            yb = {
                key: torch.as_tensor(val[batch_idx], dtype=torch.float32, device=device)
                for key, val in y_tr.items()
            }
            optimizer.zero_grad()
            loss, _ = _loss_from_outputs(_model_outputs(model, Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        tr_loss = _eval_loss_batched(model, X_tr, y_tr, device)
        va_loss = _eval_loss_batched(model, X_va, y_va, device)
        history.append({'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': va_loss, 'batch_loss': np.mean(batch_losses)})
        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs}  train={tr_loss:.6f}  val={va_loss:.6f}')

    return pd.DataFrame(history)


print('Device:', DEVICE)
print(f'Input size: {X_tr_seq.shape[2]}')


Device: mps
Input size: 17


## 8. Train

In [12]:
model = RankingConfidenceLSTM(
    input_size  = X_tr_seq.shape[2],
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
)

print(f'Training on {X_tr_seq.shape[0]:,} sequences for {EPOCHS} epochs ...')
history = train_lstm(
    model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
)

best_val = history['val_loss'].min()
print(f'\nBest val composite loss: {best_val:.8f}')
history.tail()


Training on 84,198 sequences for 30 epochs ...
  epoch   5/30  train=0.001719  val=0.001747
  epoch  10/30  train=0.001087  val=0.001188
  epoch  15/30  train=0.001052  val=0.001173
  epoch  20/30  train=0.001017  val=0.001144
  epoch  25/30  train=0.001043  val=0.001250
  epoch  30/30  train=0.000980  val=0.001258

Best val MSE: 0.00114120


,epoch,train_loss,val_loss
25,26,0.001173,0.001372
26,27,0.001035,0.001253
27,28,0.001047,0.001290
28,29,0.001069,0.001317
29,30,0.000980,0.001258


In [13]:
# Validation Diagnostics
from scipy import stats
from sklearn.metrics import roc_auc_score

print('=' * 60)
print('VALIDATION DIAGNOSTICS')
print('=' * 60)

val_outputs = _predict_batched(model, X_va_seq, DEVICE)
val_preds = val_outputs['alpha']
val_confidence = val_outputs['confidence']

naive_zero_mse  = float(np.mean(y_va_seq[TARGET_COL] ** 2))
naive_mean      = float(y_tr_seq[TARGET_COL].mean())
naive_mean_mse  = float(np.mean((y_va_seq[TARGET_COL] - naive_mean) ** 2))
lstm_val_mse    = float(np.mean((y_va_seq[TARGET_COL] - val_preds) ** 2))

print('\n-- 1. Pointwise Baseline Comparison --')
print(f'  Naive zero-predict MSE  : {naive_zero_mse:.8f}')
print(f'  Naive mean-predict MSE  : {naive_mean_mse:.8f}')
print(f'  LSTM alpha MSE          : {lstm_val_mse:.8f}')

print('\n-- 2. Directional Accuracy --')
actual_direction = np.sign(y_va_seq[TARGET_COL])
predicted_direction = np.sign(val_preds)
nonzero_mask = actual_direction != 0
dir_accuracy = float((actual_direction[nonzero_mask] == predicted_direction[nonzero_mask]).mean())
print(f'  Directional accuracy : {dir_accuracy:.4f}  ({dir_accuracy*100:.1f}%)')

print('\n-- 3. Ranking Quality --')
ic, ic_pvalue = stats.spearmanr(val_preds, y_va_seq[TARGET_COL])
ic = float(ic)
print(f'  Overall IC      : {ic:.4f}')
print(f'  IC p-value      : {ic_pvalue:.4f}')
try:
    top_auc = float(roc_auc_score(y_va_seq[TOP_QUINTILE_COL], val_confidence))
except ValueError:
    top_auc = np.nan
print(f'  Top-quintile AUC: {top_auc:.4f}')

val_dates = pd.to_datetime([m[0] for m in meta_val])
val_diag = pd.DataFrame({
    'date': val_dates,
    'predicted': val_preds,
    'confidence': val_confidence,
    'actual': y_va_seq[TARGET_COL],
    'top_quintile': y_va_seq[TOP_QUINTILE_COL],
})

print('\n-- 4. IC Stability by Month --')
ic_by_month = (
    val_diag.assign(month=lambda df: df['date'].dt.to_period('M'))
    .groupby('month')
    .apply(lambda g: float(stats.spearmanr(g['predicted'], g['actual'])[0]))
    .rename('monthly_ic')
)
positive_months = int((ic_by_month > 0).sum())
total_months = int(len(ic_by_month))
ic_hit_rate = positive_months / total_months
print(f'  Mean monthly IC   : {ic_by_month.mean():.4f}')
print(f'  Std monthly IC    : {ic_by_month.std():.4f}')
print(f'  IC hit rate       : {positive_months}/{total_months} positive ({ic_hit_rate:.1%})')
print(ic_by_month.to_string())

print('\n-- 5. Regime Diagnostics --')
regime_lookup = target_frame[['date', 'high_vol_regime', 'drawdown_regime', 'rebound_regime']].drop_duplicates('date')
regime_df = val_diag.merge(regime_lookup, on='date', how='left')
for regime_col in ['high_vol_regime', 'drawdown_regime', 'rebound_regime']:
    grp = regime_df.loc[regime_df[regime_col] == 1.0]
    if len(grp) < 20:
        continue
    regime_ic, _ = stats.spearmanr(grp['predicted'], grp['actual'])
    print(f'  {regime_col:<16s}: IC = {float(regime_ic): .4f}  n={len(grp):,}')

print('\n-- 6. Confidence Spread Check --')
val_diag['confidence_bucket'] = pd.qcut(val_diag['confidence'], q=5, labels=False, duplicates='drop')
conf_summary = val_diag.groupby('confidence_bucket').agg(
    avg_confidence=('confidence', 'mean'),
    avg_actual_alpha=('actual', 'mean'),
    top_quintile_rate=('top_quintile', 'mean'),
    n=('actual', 'size'),
)
print(conf_summary.to_string())

_diagnostic_metrics = {
    'val_alpha_mse': lstm_val_mse,
    'val_directional_accuracy': dir_accuracy,
    'val_ic': ic,
    'val_ic_hit_rate_monthly': ic_hit_rate,
    'val_top_quintile_auc': top_auc if np.isfinite(top_auc) else 0.0,
}

print('\n' + '=' * 60)
print('DIAGNOSTIC SUMMARY')
print('=' * 60)
print(f'  Alpha MSE              : {lstm_val_mse:.8f}')
print(f'  Directional accuracy   : {dir_accuracy*100:.1f}%')
print(f'  IC overall             : {ic:.4f}')
print(f'  IC hit rate monthly    : {ic_hit_rate:.1%}')
print(f'  Top-quintile AUC       : {top_auc:.4f}')
print('=' * 60)


VALIDATION DIAGNOSTICS

── 1. Naive Baseline Comparison ──
  Naive zero-predict MSE  : 0.00108578
  Naive mean-predict MSE  : 0.00108509
  LSTM val MSE (best)     : 0.00114120

  ✗ LSTM does NOT beat zero-predict — no useful signal detected
  ✗ LSTM does NOT beat mean-predict — no useful signal detected

── 2. Directional Accuracy ──
  Directional accuracy : 0.4898  (49.0%)
  ✗ At or below random (50%) — no directional signal

── 3. Information Coefficient (IC) ──
  IC (Spearman rank corr) : -0.0282
  IC p-value              : 0.0023
  ✗ IC near zero — model cannot rank stocks meaningfully

── 4. IC Stability by Month ──
  Mean monthly IC   : -0.0246
  Std monthly IC    : 0.0952
  IC hit rate       : 5/11 months positive  (45.5%)

month
2021-02    0.084223
2021-03   -0.109622
2021-04   -0.038982
2021-05   -0.050931
2021-06   -0.151474
2021-07    0.098339
2021-08    0.084641
2021-09   -0.167294
2021-10   -0.065956
2021-11    0.043559
2021-12    0.002896
Freq: M

  ✗ IC positive in only 

---------------------------------------------------------------------------------

### Checkpoint saving
Save Model Weights + `log_model_submission`

In [14]:
# ── Save model checkpoint + metadata ──────────────────────────────────────────
import json
import torch

checkpoint_dir = artifact_dir / 'model'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Single bundled checkpoint — contains everything needed to reload the model
checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
torch.save(
    {
        'state_dict'          : model.state_dict(),
        'feature_names'       : ALL_FEATURES,
        'horizon'             : HORIZON,
        'rebalance_frequency' : 'every_5_trading_days',
        'model_config': {
            'architecture' : 'RankingConfidenceLSTM',
            'input_dim'    : len(ALL_FEATURES),
            'hidden_dim'   : HIDDEN_SIZE,
            'num_layers'   : NUM_LAYERS,
            'dropout'      : DROPOUT,
        },
        'preprocessing': {
            'scaler' : 'train_mean_std',
            'means'  : train_mean.to_dict(),
            'stds'   : train_std.to_dict(),
        },
        'training': {
            'epochs'       : EPOCHS,
            'batch_size'   : BATCH_SIZE,
            'lr'           : LR,
            'seed'         : SEED,
            'best_val_mse' : round(float(best_val), 8),
        },
        'data': {
            'dataset_name'   : dataset.name if hasattr(dataset, 'name') else str(dataset),
            'tickers'        : list(spec.tickers),
            'benchmark'      : str(spec.benchmark_ticker),
            'train_end'      : str(spec.train_end),
            'val_end'        : str(spec.val_end),
            'test_start'     : str(spec.test_start),
            'target'         : TARGET_COL,
            'seq_len'        : SEQ_LEN,
            'custom_features': CUSTOM_FEATURES,
            'regime_features': REGIME_FEATURES,
            'cross_sectional_feature_count': len(CS_FEATURES),
        },
        'source_notebook' : 'baseline_lstm.ipynb',
        'run_name'        : RUN_NAME,
        'author'          : 'Minho',
        'loss_weights': {
            'alpha': ALPHA_LOSS_WEIGHT,
            'rank': RANK_LOSS_WEIGHT,
            'top_quintile': TOP_QUINTILE_WEIGHT,
            'volatility': VOL_LOSS_WEIGHT,
            'downside': DOWNSIDE_LOSS_WEIGHT,
        },
        'portfolio_config': {
            'sleeve_sizes': SLEEVE_SIZES,
            'broad_top_fraction': BROAD_TOP_FRACTION,
            'confidence_top_fraction': CONFIDENCE_TOP_FRACTION,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'max_weight': MAX_WEIGHT,
            'min_active_names': MIN_ACTIVE_NAMES,
            'no_trade_band': NO_TRADE_BAND,
        },
    },
    checkpoint_path,
)
print(f'Bundled checkpoint saved → {checkpoint_path}')

# Reconstruction recipe
recipe = """
# How to reproduce Minho baseline LSTM predictions from scratch
#
# Requirements: repo installed, venv312 active, MLflow artifacts downloaded
#
# Steps:
#   1. Load the bundled checkpoint
#      bundle = torch.load('lstm_checkpoint.pt', map_location='cpu')
#
#   2. Rebuild model architecture
#      model = BaselineLSTM(
#          input_size  = bundle['model_config']['input_dim'],
#          hidden_size = bundle['model_config']['hidden_dim'],
#          num_layers  = bundle['model_config']['num_layers'],
#          dropout     = bundle['model_config']['dropout'],
#      )
#      model.load_state_dict(bundle['state_dict'])
#      model.eval()
#
#   3. Load preprocessing
#      train_mean = pd.Series(bundle['preprocessing']['means'])
#      train_std  = pd.Series(bundle['preprocessing']['stds'])
#
#   4. Load prices and call predict_from_prices()
#      prices = load_prices(bundle['data']['dataset_name'])
#      predictions = predict_from_prices(
#          model=model, prices=prices,
#          train_mean=train_mean, train_std=train_std,
#          feature_names=bundle['feature_names'],
#          seq_len=bundle['data']['seq_len'],
#          horizon=bundle['horizon'],
#      )
"""
recipe_path = checkpoint_dir / 'REPRODUCE.py'
recipe_path.write_text(recipe)
print(f'Reproduction recipe saved → {recipe_path}')

print('\nAll model artifacts saved:')
for p in sorted(checkpoint_dir.iterdir()):
    print(f'  {p.name}')

Bundled checkpoint saved → /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/model/lstm_checkpoint.pt
Reproduction recipe saved → /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/model/REPRODUCE.py

All model artifacts saved:
  REPRODUCE.py
  lstm_checkpoint.pt
  lstm_weights.pt
  model_metadata.json
  norm_stats.json


---------------------------------------------------------------------------------

## 9. Predictions

In [15]:
# Generate predictions — SPY explicitly excluded via TRADABLE_TICKERS
predictions = predict_from_prices(
    model         = model,
    prices        = prices,
    train_mean    = train_mean,
    train_std     = train_std,
    feature_names = ALL_FEATURES,
    seq_len       = SEQ_LEN,
    horizon       = HORIZON,
    tickers       = TRADABLE_TICKERS,
)

# Filter to test split dates only
test_dates = set(test['date'].unique())
predictions = predictions[
    predictions['date'].isin(test_dates)
].reset_index(drop=True)

# Filter to rebalance dates only — must match portfolio weight dates
# so MLflow predictions and weights are consistent
all_pred_dates = sorted(predictions['date'].unique())
rebal_dates    = set(pd.DatetimeIndex(all_pred_dates)[::REBALANCE_FREQ])
predictions    = predictions[
    predictions['date'].isin(rebal_dates)
].reset_index(drop=True)

# Validate
assert BENCHMARK_TICKER not in predictions['ticker'].values, \
    f'{BENCHMARK_TICKER} found in predictions — exclusion failed'

print('Predictions shape :', predictions.shape)
print('Rebalance dates   :', len(rebal_dates))
print('SPY in predictions:', BENCHMARK_TICKER in predictions['ticker'].values)
predictions.head()

Predictions shape : (9799, 4)
Rebalance dates   : 200
SPY in predictions: False


,date,ticker,horizon,expected_return
0,2022-01-03,AAPL,5,0.005763
1,2022-01-03,ABBV,5,0.016938
2,2022-01-03,ADBE,5,0.008293
3,2022-01-03,AMD,5,-0.017889
4,2022-01-03,AMZN,5,0.006813


## 10. Confidence-Aware Portfolio Weights

The baseline rank-long-only layer spread weights across the universe. v3 builds several candidate sleeves and a dynamic confidence sleeve. The final selected portfolio uses broad exposure by default, then concentrates when confidence and predicted spread justify it. Every candidate gets a hard max-weight projection after selection.


In [16]:
# Confidence-aware portfolio construction
from portfolio_toolkit.contracts import PortfolioWeights as _PW


def _effective_names(weights: pd.Series) -> float:
    w = weights[weights > 0].astype(float)
    if w.empty:
        return 0.0
    return float(1.0 / np.square(w).sum())


def _project_capped_simplex(weights: pd.Series, max_weight: float) -> pd.Series:
    """Project long-only weights onto sum=1 with a hard per-name cap."""
    w = weights.clip(lower=0.0).astype(float).copy()
    if max_weight <= 0:
        raise ValueError('max_weight must be positive')
    if max_weight * len(w) < 1.0 - 1e-12:
        raise ValueError('max_weight is too small for the number of active names')
    if w.sum() <= 0:
        w[:] = 1.0

    result = pd.Series(0.0, index=w.index, dtype=float)
    remaining = w.index.copy()
    residual = 1.0

    for _ in range(len(w) + 1):
        if len(remaining) == 0 or residual <= 1e-15:
            break
        base = w.loc[remaining]
        if base.sum() <= 0:
            alloc = pd.Series(residual / len(remaining), index=remaining)
        else:
            alloc = base / base.sum() * residual
        over = alloc > max_weight
        if not over.any():
            result.loc[remaining] = alloc
            residual = 0.0
            break
        capped_names = alloc.index[over]
        result.loc[capped_names] = max_weight
        residual -= max_weight * len(capped_names)
        remaining = alloc.index[~over]

    slack = 1.0 - result.sum()
    if abs(slack) > 1e-10:
        capacity = (max_weight - result).clip(lower=0.0)
        if capacity.sum() > 0:
            result += capacity / capacity.sum() * slack
    result = result.clip(lower=0.0, upper=max_weight)
    if abs(result.sum() - 1.0) > 1e-8:
        under = result < max_weight - 1e-12
        if under.any():
            result.loc[under] += (1.0 - result.sum()) * (result.loc[under] / result.loc[under].sum())
    return result.fillna(0.0)


def _softmax_weights(scores: pd.Series, temperature: float) -> pd.Series:
    values = scores.astype(float).replace([np.inf, -np.inf], np.nan).fillna(scores.median())
    values = values - values.max()
    temp = max(float(temperature), 1e-6)
    exp_values = np.exp(values / temp)
    return pd.Series(exp_values, index=scores.index) / exp_values.sum()


def _apply_no_trade_band(current: pd.Series, previous: pd.Series | None, band: float) -> pd.Series:
    if previous is None or band <= 0:
        return current
    previous = previous.reindex(current.index).fillna(0.0)
    adjusted = current.copy()
    keep = (current - previous).abs() < band
    adjusted[keep] = previous[keep]
    if adjusted.sum() <= 0:
        return current
    return adjusted / adjusted.sum()


def build_confidence_portfolio(
    predictions: pd.DataFrame,
    *,
    strategy_name: str,
    mode: str = 'dynamic',
    top_k: int | None = None,
    broad_top_fraction: float = BROAD_TOP_FRACTION,
    confidence_top_fraction: float = CONFIDENCE_TOP_FRACTION,
    confidence_threshold: float = CONFIDENCE_THRESHOLD,
    max_weight: float = MAX_WEIGHT,
    min_active_names: int = MIN_ACTIVE_NAMES,
    no_trade_band: float = NO_TRADE_BAND,
    softmax_temperature: float = SOFTMAX_TEMPERATURE,
    risk_aversion: float = RISK_AVERSION,
    downside_aversion: float = DOWNSIDE_AVERSION,
) -> tuple[_PW, pd.DataFrame]:
    validated = validate_prediction_frame(predictions)
    tickers = sorted(validated['ticker'].unique().tolist())
    weights = pd.DataFrame(0.0, index=pd.DatetimeIndex(sorted(validated['date'].unique())), columns=tickers)
    weights.index.name = 'date'
    diagnostics = []
    prev = None

    for date_value, frame_date in validated.groupby('date', sort=True):
        frame_date = frame_date.copy()
        n = len(frame_date)
        if 'confidence' not in frame_date.columns:
            frame_date['confidence'] = 0.5
        else:
            frame_date['confidence'] = frame_date['confidence'].fillna(0.5)
        if 'expected_downside' not in frame_date.columns:
            frame_date['expected_downside'] = 0.0
        else:
            frame_date['expected_downside'] = frame_date['expected_downside'].fillna(0.0)
        frame_date['rank_pct'] = frame_date['expected_return'].rank(pct=True, method='average')
        if 'expected_volatility' not in frame_date.columns:
            frame_date['expected_volatility'] = 0.0
        else:
            frame_date['expected_volatility'] = frame_date['expected_volatility'].fillna(0.0)
        frame_date['risk_penalty'] = np.exp(
            -risk_aversion * frame_date['expected_volatility'].clip(lower=0.0)
            -downside_aversion * frame_date['expected_downside'].clip(lower=0.0)
        )
        frame_date['portfolio_score'] = (
            frame_date['rank_pct']
            * (0.50 + frame_date['confidence'].clip(0.0, 1.0))
            * frame_date['risk_penalty']
        )

        spread = float(frame_date['expected_return'].quantile(0.90) - frame_date['expected_return'].quantile(0.10))
        top_conf = float(frame_date.nlargest(max(1, int(np.ceil(n * 0.20))), 'expected_return')['confidence'].mean())

        if mode == 'top_k':
            k = min(n, max(1, int(top_k or n)))
            concentration_on = True
        elif mode == 'broad':
            k = min(n, max(min_active_names, int(np.ceil(n * broad_top_fraction))))
            concentration_on = False
        else:
            broad_k = min(n, max(min_active_names, int(np.ceil(n * broad_top_fraction))))
            concentrated_k = min(n, max(min_active_names, int(np.ceil(n * confidence_top_fraction))))
            concentration_on = top_conf >= confidence_threshold
            k = concentrated_k if concentration_on else broad_k

        row_max_weight = max(max_weight, 1.0 / k + 1e-12) if mode == 'top_k' else max_weight

        chosen = frame_date.nlargest(k, 'portfolio_score').copy()
        raw = _softmax_weights(chosen.set_index('ticker')['portfolio_score'], softmax_temperature)
        row = pd.Series(0.0, index=tickers)
        row.loc[raw.index] = _project_capped_simplex(raw, row_max_weight)
        row = _apply_no_trade_band(row, prev, no_trade_band)
        if (row > 1e-12).sum() < min_active_names:
            active = set(row[row > 1e-12].index)
            needed = max(0, min_active_names - len(active))
            fill_names = [
                t for t in frame_date.nlargest(min_active_names, 'portfolio_score')['ticker']
                if t not in active
            ][:needed]
            if fill_names:
                row.loc[fill_names] = np.maximum(row.loc[fill_names], 1e-6)
        row = _project_capped_simplex(row, row_max_weight)
        weights.loc[pd.Timestamp(date_value), row.index] = row.values
        prev = row

        diagnostics.append({
            'date': pd.Timestamp(date_value),
            'mode': mode,
            'selected_names': int((row > 0).sum()),
            'effective_names': _effective_names(row),
            'max_weight': float(row.max()),
            'effective_max_weight_cap': float(row_max_weight),
            'top_confidence': top_conf,
            'predicted_spread_90_10': spread,
            'concentration_on': bool(concentration_on),
        })

    clean_weights = validate_weights_frame(weights)
    return _PW(
        weights=clean_weights,
        dataset_name=DATASET_NAME.name if hasattr(DATASET_NAME, 'name') else DATASET_NAME,
        strategy_name=strategy_name,
        metadata={
            'mode': mode,
            'max_weight': max_weight,
            'min_active_names': min_active_names,
            'no_trade_band': no_trade_band,
            'confidence_threshold': confidence_threshold,
            'softmax_temperature': softmax_temperature,
            'risk_aversion': risk_aversion,
            'downside_aversion': downside_aversion,
        },
    ), pd.DataFrame(diagnostics)


portfolio_candidates = {}
portfolio_diagnostics = {}

portfolio_candidates['broad'], portfolio_diagnostics['broad'] = build_confidence_portfolio(
    predictions, strategy_name=f'{RUN_NAME}_broad', mode='broad'
)
for k in SLEEVE_SIZES:
    portfolio_candidates[f'top_{k}'], portfolio_diagnostics[f'top_{k}'] = build_confidence_portfolio(
        predictions, strategy_name=f'{RUN_NAME}_top_{k}', mode='top_k', top_k=k,
        min_active_names=min(k, len(TRADABLE_TICKERS)), no_trade_band=0.0,
    )
portfolio_candidates['dynamic'], portfolio_diagnostics['dynamic'] = build_confidence_portfolio(
    predictions, strategy_name=RUN_NAME, mode='dynamic'
)

clean_portfolio = portfolio_candidates['dynamic']
portfolio = clean_portfolio
validated_weights = validate_weights_frame(clean_portfolio.weights)
selected_diag = portfolio_diagnostics['dynamic']

print('Selected portfolio : dynamic confidence sleeve')
print('Weights shape      :', validated_weights.shape)
print('Avg active names   :', selected_diag['selected_names'].mean())
print('Avg effective names:', selected_diag['effective_names'].mean())
print('Avg max weight     :', selected_diag['max_weight'].mean())
print('Concentration days :', selected_diag['concentration_on'].mean())
print('Row sums OK        :', np.allclose(validated_weights.sum(axis=1), 1.0))
selected_diag.tail()


Weights shape  : (40, 49)
Max weight     : 0.04000000000000001
Rebal dates    : 40
Row sums OK    : True


----------------------------------------------------------------------------------------

## 11. Backtest

In [17]:
# ── Backtest ──────────────────────────────────────────────────────────
import bt
from portfolio_toolkit.backtest import (
    _pivot_prices,
    _align_weights_to_prices,
    _mask_unavailable_weights,
    _make_bt_strategy,
    _commission_fn,
    _compute_turnover,
)
from portfolio_toolkit.reporting import build_metrics
from portfolio_toolkit.contracts import BacktestResult

# ── Step 1: Strict price cleaning ─────────────────────────────────────────────
price_wide_raw    = _pivot_prices(prices)
price_wide_filled = price_wide_raw.ffill(limit=5).bfill(limit=5)

clean_mask  = price_wide_filled.notna().all(axis=0)
price_wide  = price_wide_filled.loc[:, clean_mask]

dropped = sorted(set(price_wide_raw.columns) - set(price_wide.columns))
print(f'Tickers dropped (missing prices) : {len(dropped)}')
if dropped:
    print(f'  {dropped}')

available_tickers = [t for t in TRADABLE_TICKERS if t in price_wide.columns]
print(f'Clean tickers for backtest       : {len(available_tickers)}')

assert price_wide[available_tickers].isna().sum().sum() == 0, \
    'NaNs still present — bt will crash'
print('Price matrix is clean — no NaNs detected.')

# ── Step 2: Align and mask strategy weights ────────────────────────────────────
raw_weights     = validate_weights_frame(clean_portfolio.weights)
aligned_weights = _align_weights_to_prices(
    raw_weights.loc[:, [t for t in available_tickers if t in raw_weights.columns]],
    price_wide.index,
)
aligned_weights = _mask_unavailable_weights(
    aligned_weights, price_wide.loc[:, available_tickers]
)

# ── Step 3: Equal weight baseline (no SPY) ────────────────────────────────────
eq_value      = 1.0 / len(available_tickers)
equal_weights = pd.DataFrame(
    eq_value,
    index   = aligned_weights.index,
    columns = available_tickers,
    dtype   = float,
)
equal_weights.index.name = 'date'
equal_weights = _align_weights_to_prices(equal_weights, price_wide.index)
equal_weights = _mask_unavailable_weights(
    equal_weights, price_wide.loc[:, available_tickers]
)

# ── Step 4: SPY benchmark ──────────────────────────────────────────────────────
spy_bench = pd.DataFrame(
    {'SPY': 1.0},
    index = aligned_weights.index,
)
spy_bench.index.name = 'date'
spy_bench = _align_weights_to_prices(spy_bench, price_wide.index)

cost_fn = _commission_fn(spec.cost_bps)

# ── Step 5: Run bt backtests ───────────────────────────────────────────────────
print('\nRunning backtests ...')
backtests = [
    bt.Backtest(
        _make_bt_strategy(RUN_NAME, aligned_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('SPY', spy_bench),
        price_wide[['SPY']],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('equal_weight', equal_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
]

bt_result = bt.run(*backtests)

# ── Step 6: Assemble BacktestResult ───────────────────────────────────────────
nav      = bt_result.prices[RUN_NAME].rename('nav')
returns  = nav.pct_change().fillna(0.0).rename('returns')
turnover = _compute_turnover(aligned_weights)

benchmark_returns = pd.DataFrame(index=nav.index)
for col in bt_result.prices.columns:
    if col == RUN_NAME:
        continue
    benchmark_returns[col] = bt_result.prices[col].pct_change().fillna(0.0)

result = BacktestResult(
    strategy_name    = RUN_NAME,
    dataset_name     = dataset.name if hasattr(dataset, 'name') else str(dataset),
    weights          = aligned_weights,
    nav              = nav,
    returns          = returns,
    turnover         = turnover,
    benchmark_returns= benchmark_returns,
    metrics          = {},
)
result.metrics = build_metrics(result)

print('Backtest complete.')
print()
for k, v in sorted(result.metrics.items()):
    print(f'  {k:<35s}: {v:.6f}')

Tickers dropped (missing prices) : 0
Clean tickers for backtest       : 49
Price matrix is clean — no NaNs detected.

Running backtests ...
Backtest complete.

  annual_excess_return_vs_benchmark  : 0.076444
  annual_return                      : 0.184755
  annual_volatility                  : 0.179582
  average_turnover                   : 0.325510
  benchmark_annual_return            : 0.108312
  benchmark_annual_volatility        : 0.179691
  benchmark_max_drawdown             : -0.244964
  benchmark_sharpe                   : 0.602767
  benchmark_total_return             : 0.507582
  calmar                             : 0.885351
  evaluation_trading_days            : 1003.000000
  evaluation_years                   : 3.991786
  excess_return_vs_benchmark         : 0.459897
  max_drawdown                       : -0.208680
  sharpe                             : 1.028810
  sharpe_vs_benchmark                : 0.426043
  sortino                            : 1.424992
  total_return     

------------------------------------------------------------------------------------------

In [19]:
# Benchmark and sleeve comparison table
from math import sqrt
from IPython.display import display

print('=' * 80)
print('BENCHMARK + SLEEVE COMPARISON')
print('=' * 80)


def compute_metrics(nav: pd.Series, returns: pd.Series) -> dict:
    total_return  = float(nav.iloc[-1] / nav.iloc[0] - 1.0)
    n_days        = max(len(returns), 1)
    annual_return = float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
    annual_vol    = float(returns.std(ddof=0) * sqrt(252))
    sharpe        = annual_return / annual_vol if annual_vol > 0 else 0.0
    downside      = returns[returns < 0]
    sortino_vol   = float(downside.std(ddof=0) * sqrt(252)) if len(downside) else 0.0
    sortino       = annual_return / sortino_vol if sortino_vol > 0 else 0.0
    drawdown      = nav / nav.cummax() - 1.0
    max_dd        = float(drawdown.min())
    calmar        = annual_return / abs(max_dd) if max_dd < 0 else 0.0
    return {
        'Total Return' : total_return,
        'Annual Return': annual_return,
        'Annual Vol'   : annual_vol,
        'Sharpe'       : sharpe,
        'Sortino'      : sortino,
        'Max Drawdown' : max_dd,
        'Calmar'       : calmar,
    }


# Evaluate sleeves on the same cleaned price matrix used by the selected backtest.
sleeve_backtests = []
sleeve_weight_cache = {}
for name, candidate in portfolio_candidates.items():
    candidate_weights = validate_weights_frame(candidate.weights)
    candidate_aligned = _align_weights_to_prices(
        candidate_weights.loc[:, [t for t in available_tickers if t in candidate_weights.columns]],
        price_wide.index,
    )
    candidate_aligned = _mask_unavailable_weights(
        candidate_aligned, price_wide.loc[:, available_tickers]
    )
    sleeve_weight_cache[name] = candidate_aligned
    sleeve_backtests.append(
        bt.Backtest(
            _make_bt_strategy(name, candidate_aligned),
            price_wide[available_tickers],
            commissions=cost_fn,
            integer_positions=False,
        )
    )

sleeve_bt_result = bt.run(*sleeve_backtests)
sleeve_rows = []
for name in portfolio_candidates:
    sleeve_nav = sleeve_bt_result.prices[name].rename('nav')
    sleeve_returns = sleeve_nav.pct_change().fillna(0.0)
    m = compute_metrics(sleeve_nav / sleeve_nav.iloc[0], sleeve_returns)
    diag = portfolio_diagnostics[name]
    sleeve_rows.append({
        'portfolio': name,
        'total_return': m['Total Return'],
        'annual_return': m['Annual Return'],
        'sharpe': m['Sharpe'],
        'sortino': m['Sortino'],
        'max_drawdown': m['Max Drawdown'],
        'avg_active_names': diag['selected_names'].mean(),
        'avg_effective_names': diag['effective_names'].mean(),
        'avg_max_weight': diag['max_weight'].mean(),
    })

sleeve_summary = pd.DataFrame(sleeve_rows).sort_values('sharpe', ascending=False)
print('\nSleeve comparison')
display(sleeve_summary.style.format({
    'total_return': '{:.1%}',
    'annual_return': '{:.1%}',
    'sharpe': '{:.3f}',
    'sortino': '{:.3f}',
    'max_drawdown': '{:.1%}',
    'avg_active_names': '{:.1f}',
    'avg_effective_names': '{:.1f}',
    'avg_max_weight': '{:.2%}',
}))

# Existing dynamic-vs-benchmark comparison.
spy_returns = result.benchmark_returns['SPY'] if 'SPY' in result.benchmark_returns.columns else result.benchmark_returns.iloc[:, 0]
eq_returns = result.benchmark_returns['equal_weight'] if 'equal_weight' in result.benchmark_returns.columns else None

spy_nav = (1.0 + spy_returns).cumprod()
spy_nav = spy_nav / spy_nav.iloc[0]
if eq_returns is not None:
    eq_nav = (1.0 + eq_returns).cumprod()
    eq_nav = eq_nav / eq_nav.iloc[0]
else:
    eq_returns = spy_returns
    eq_nav = spy_nav
    print('Warning: equal_weight not found in benchmark_returns, using SPY as proxy')

lstm_metrics = compute_metrics(result.nav / result.nav.iloc[0], result.returns)
spy_metrics  = compute_metrics(spy_nav, spy_returns)
eq_metrics   = compute_metrics(eq_nav, eq_returns)

def excess(a, b): return a - b
lstm_vs_spy = {k: excess(lstm_metrics[k], spy_metrics[k]) for k in lstm_metrics}
lstm_vs_eq  = {k: excess(lstm_metrics[k], eq_metrics[k])  for k in lstm_metrics}

pct_metrics = {'Total Return', 'Annual Return', 'Annual Vol', 'Max Drawdown'}
strategies  = ['Dynamic LSTM', 'SPY Buy & Hold', 'Equal Weight']
metrics_all = [lstm_metrics, spy_metrics, eq_metrics]

header = f'{"Metric":<22s}' + ''.join(f'{s:>18s}' for s in strategies)
print(f'\n{header}')
print('-' * (22 + 18 * len(strategies)))
for metric in lstm_metrics:
    row = f'{metric:<22s}'
    for m in metrics_all:
        val = m[metric]
        row += f'{val:>17.1%} ' if metric in pct_metrics else f'{val:>17.4f} '
    print(row)

for label, diff in [('Excess vs SPY', lstm_vs_spy), ('Excess vs Equal Weight', lstm_vs_eq)]:
    print(f'\n{label}')
    print('-' * 40)
    for metric in ['Total Return', 'Annual Return', 'Sharpe', 'Max Drawdown']:
        val  = diff[metric]
        fmt  = f'{val:>+.1%}' if metric in pct_metrics else f'{val:>+.4f}'
        flag = 'OK' if val > 0 else 'CHECK'
        print(f'  {flag:<5s} {metric:<20s}: {fmt}')

print('\n-- Portfolio Shape --')
print(f'  Avg daily turnover      : {result.turnover.mean():.4f}')
print(f'  Avg active names        : {selected_diag["selected_names"].mean():.1f}')
print(f'  Avg effective names     : {selected_diag["effective_names"].mean():.1f}')
print(f'  Avg max weight          : {selected_diag["max_weight"].mean():.2%}')
print(f'  Concentration-on days   : {selected_diag["concentration_on"].mean():.1%}')

_comparison_metrics_cache = {
    'lstm_annual_return'          : lstm_metrics['Annual Return'],
    'lstm_sharpe'                 : lstm_metrics['Sharpe'],
    'lstm_max_drawdown'           : lstm_metrics['Max Drawdown'],
    'spy_annual_return'           : spy_metrics['Annual Return'],
    'spy_sharpe'                  : spy_metrics['Sharpe'],
    'eq_annual_return'            : eq_metrics['Annual Return'],
    'eq_sharpe'                   : eq_metrics['Sharpe'],
    'excess_annual_return_vs_spy' : lstm_vs_spy['Annual Return'],
    'excess_sharpe_vs_spy'        : lstm_vs_spy['Sharpe'],
    'excess_annual_return_vs_eq'  : lstm_vs_eq['Annual Return'],
    'excess_sharpe_vs_eq'         : lstm_vs_eq['Sharpe'],
    'avg_active_names'            : float(selected_diag['selected_names'].mean()),
    'avg_effective_names'         : float(selected_diag['effective_names'].mean()),
    'avg_max_weight'              : float(selected_diag['max_weight'].mean()),
    'concentration_day_rate'      : float(selected_diag['concentration_on'].mean()),
}

for _, row in sleeve_summary.iterrows():
    prefix = f"sleeve_{row['portfolio']}"
    _comparison_metrics_cache[f'{prefix}_annual_return'] = float(row['annual_return'])
    _comparison_metrics_cache[f'{prefix}_sharpe'] = float(row['sharpe'])
    _comparison_metrics_cache[f'{prefix}_max_drawdown'] = float(row['max_drawdown'])

print('\n' + '=' * 80)
print('BENCHMARK + SLEEVE COMPARISON COMPLETE')
print('=' * 80)


BENCHMARK COMPARISON

Metric                      LSTM (Minho)    SPY Buy & Hold      Equal Weight
----------------------------------------------------------------------------
Total Return                      96.6%             50.8%             72.8% 
Annual Return                      5.8%              3.5%              4.7% 
Annual Vol                        10.4%             10.4%              9.8% 
Sharpe                           0.5590            0.3359            0.4776 
Sortino                          0.4469            0.2673            0.3749 
Max Drawdown                     -20.9%            -24.6%            -21.9% 
Calmar                           0.2781            0.1419            0.2135 

Excess vs SPY
----------------------------------------
  ✓ Total Return        : +45.8%
  ✓ Annual Return       : +2.3%
  ✓ Sharpe              : +0.2231
  ✓ Max Drawdown        : +3.7%

Excess vs Equal Weight
----------------------------------------
  ✓ Total Return        : +23.8%


In [ ]:
# ADVERSARIAL STRESS: try to find how the strategy loses money
print('=' * 80)
print('ADVERSARIAL STRESS: HOW DO WE LOSE MONEY?')
print('=' * 80)

stress_df = pd.DataFrame({
    'strategy_return': result.returns,
    'benchmark_return': spy_returns.reindex(result.returns.index).fillna(0.0),
    'turnover': result.turnover.reindex(result.returns.index).fillna(0.0),
})
stress_df['active_names'] = (aligned_weights > 1e-12).sum(axis=1).reindex(stress_df.index).ffill()
stress_df['max_weight'] = aligned_weights.max(axis=1).reindex(stress_df.index).ffill()
stress_df['effective_names'] = aligned_weights.apply(_effective_names, axis=1).reindex(stress_df.index).ffill()
stress_df['strategy_drawdown'] = result.nav / result.nav.cummax() - 1.0

worst_strategy_days = stress_df.nsmallest(10, 'strategy_return')
worst_benchmark_days = stress_df.nsmallest(10, 'benchmark_return')
high_turnover_days = stress_df.nlargest(10, 'turnover')

print('\nWorst strategy days')
display(worst_strategy_days.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'strategy_drawdown', 'max_weight']))

print('\nWorst benchmark days: did the strategy defend capital?')
benchmark_stress = worst_benchmark_days.assign(excess_vs_benchmark=lambda df: df['strategy_return'] - df['benchmark_return'])
display(benchmark_stress.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'excess_vs_benchmark', 'strategy_drawdown', 'max_weight']))

print('\nHighest turnover days')
display(high_turnover_days.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'turnover', 'strategy_drawdown', 'max_weight']))

stress_summary = {
    'worst_strategy_day': float(stress_df['strategy_return'].min()),
    'avg_return_on_worst_benchmark_10_days': float(worst_benchmark_days['strategy_return'].mean()),
    'avg_excess_on_worst_benchmark_10_days': float((worst_benchmark_days['strategy_return'] - worst_benchmark_days['benchmark_return']).mean()),
    'avg_active_names': float(stress_df['active_names'].mean()),
    'avg_effective_names': float(stress_df['effective_names'].mean()),
    'avg_max_weight': float(stress_df['max_weight'].mean()),
    'avg_turnover': float(stress_df['turnover'].mean()),
}

print('\nStress summary')
for k, v in stress_summary.items():
    fmt = f'{v:.2%}' if any(token in k for token in ['day', 'return', 'excess', 'weight', 'turnover']) else f'{v:.2f}'
    print(f'  {k:<42s}: {fmt}')

if '_comparison_metrics_cache' in dir():
    _comparison_metrics_cache.update({f'stress_{k}': v for k, v in stress_summary.items()})
print('=' * 80)


------------------------------------------------------------------------------------------

## 12. Write Artifacts (QuantStats report + parquet files)

In [20]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

  ✓ weights             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/weights.parquet
  ✓ nav                 : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/nav.parquet
  ✓ returns             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/returns.parquet
  ✓ turnover            : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/turnover.parquet
  ✓ benchmarks          : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/benchmarks.parquet
  ✓ metrics             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/metrics.json
  ✓ metrics_table       : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/metrics_table.parquet
  ✓ quantstats_report   : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/quantstats.html


## 13. Log to MLflow as `Minho_Baseline_LSTM`

In [30]:
import mlflow
from portfolio_toolkit.tracking import log_model_submission

mlflow_layout = init_mlflow(repo_root)
print('Tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name     = RUN_NAME,
    dataset_name = dataset,        # ← pass the object, not dataset.name
    tags={
        'author'        : 'Minho',
        'model_family'  : MODEL_FAMILY,
        'workflow'      : 'ranking_confidence_lstm_v3',
        'horizon'       : str(HORIZON),
        'dataset'       : dataset.name,
        'project'       : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    # Params
    mlflow.log_params({
        'run_name'            : RUN_NAME,
        'dataset'             : dataset.name if hasattr(dataset, 'name') else str(DATASET_NAME),
        'horizon'             : HORIZON,
        'seq_len'             : SEQ_LEN,
        'hidden_size'         : HIDDEN_SIZE,
        'num_layers'          : NUM_LAYERS,
        'dropout'             : DROPOUT,
        'lr'                  : LR,
        'epochs'              : EPOCHS,
        'batch_size'          : BATCH_SIZE,
        'n_features'          : len(ALL_FEATURES),
        'custom_features'     : ','.join(CUSTOM_FEATURES),
        'regime_features'     : ','.join(REGIME_FEATURES),
        'n_cs_features'       : len(CS_FEATURES),
        'portfolio_builder'   : 'confidence_dynamic_sleeve',
        'best_val_mse'        : round(float(best_val), 8),
        'target_type'         : TARGET_TYPE,
        'rebalance_freq'      : REBALANCE_FREQ,
        'rebalance_frequency' : 'every_5_trading_days',   # ← string format Adam requires
        'max_weight_cap'      : MAX_WEIGHT,
        'rank_loss_weight'    : RANK_LOSS_WEIGHT,
        'top_quintile_weight' : TOP_QUINTILE_WEIGHT,
        'vol_loss_weight'     : VOL_LOSS_WEIGHT,
        'downside_loss_weight': DOWNSIDE_LOSS_WEIGHT,
        'sleeve_sizes'        : ','.join(map(str, SLEEVE_SIZES)),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'min_active_names'    : MIN_ACTIVE_NAMES,
        'no_trade_band'       : NO_TRADE_BAND,
        'training_cutoff'     : str(TRAINING_CUTOFF),
        'softmax_temperature' : SOFTMAX_TEMPERATURE,
        'risk_aversion'       : RISK_AVERSION,
        'downside_aversion'   : DOWNSIDE_AVERSION,
    })

    # Validation metrics
    mlflow.log_metric('val_mse',  round(float(best_val), 8))
    mlflow.log_metric('val_rmse', round(float(best_val ** 0.5), 8))

    # Diagnostic metrics
    if '_diagnostic_metrics' in dir():
        mlflow.log_metrics(_diagnostic_metrics)

    # Backtest metrics + report artifacts
    log_predictions(predictions)
    log_portfolio(clean_portfolio)
    log_backtest(result)

    # Benchmark comparison metrics
    if '_comparison_metrics_cache' in dir():
        mlflow.log_metrics(_comparison_metrics_cache)
        print('Benchmark comparison metrics logged.')

    # ── Model submission bundle (Adam's required format) ──────────────────
    notebook_path = repo_root / 'baseline_lstm.ipynb'

    log_model_submission(
        {'torch_model': str(checkpoint_path)},
        model_name          = RUN_NAME,
        model_family        = 'torch',
        feature_names       = ALL_FEATURES,
        target              = TARGET_COL,
        horizon             = HORIZON,
        rebalance_frequency = 'every_5_trading_days',
        preprocessing       = {'scaler': 'train_mean_std'},
        model_config        = {
            'architecture'      : 'RankingConfidenceLSTM',
            'input_dim'         : len(ALL_FEATURES),
            'hidden_dim'        : HIDDEN_SIZE,
            'num_layers'        : NUM_LAYERS,
            'portfolio_builder' : 'build_confidence_portfolio',
            'required_functions': ['build_model_features', 'predict_from_prices', 'build_confidence_portfolio'],
        },
        source_files = [str(notebook_path)],
    )
    print('Model submission bundle logged to MLflow.')

print(f'MLflow run "{RUN_NAME}" logged successfully.')
print('Artifacts visible in MLflow UI under: Artifacts → model_submission/')

Tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
Benchmark comparison metrics logged.
Model submission bundle logged to MLflow.
🏃 View run Minho_Baseline_LSTM at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/5/runs/5724af17ff454466b27fe22f89fdd2a4
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/5
MLflow run "Minho_Baseline_LSTM" logged successfully.
Artifacts visible in MLflow UI under: Artifacts → model_submission/


In [26]:
# ── Reproducibility verification ──────────────────────────────────
import torch
import numpy as np
import pandas as pd

def load_model_from_artifacts(checkpoint_dir: Path) -> tuple:
    """
    Reconstruct the trained model + preprocessing from the bundled checkpoint.

    Parameters
    ----------
    checkpoint_dir : Path containing lstm_checkpoint.pt

    Returns
    -------
    model          : BaselineLSTM in eval mode
    train_mean     : pd.Series of feature means
    train_std      : pd.Series of feature stds
    bundle         : full checkpoint dict
    """
    checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
    bundle = torch.load(checkpoint_path, map_location='cpu')

    cfg = bundle['model_config']
    reconstructed_model = BaselineLSTM(
        input_size  = cfg['input_dim'],
        hidden_size = cfg['hidden_dim'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    )
    reconstructed_model.load_state_dict(bundle['state_dict'])
    reconstructed_model.eval()

    train_mean_loaded = pd.Series(bundle['preprocessing']['means'])
    train_std_loaded  = pd.Series(bundle['preprocessing']['stds'])

    return reconstructed_model, train_mean_loaded, train_std_loaded, bundle


# ── Run the verification ───────────────────────────────────────────────────────
print('Loading model from bundled checkpoint (simulating fresh session) ...')
reconstructed_model, train_mean_loaded, train_std_loaded, loaded_bundle = \
    load_model_from_artifacts(checkpoint_dir)

print(f'  Architecture  : {loaded_bundle["model_config"]}')
print(f'  Feature count : {len(loaded_bundle["feature_names"])}')
print(f'  Horizon       : {loaded_bundle["horizon"]}')
print(f'  Rebal freq    : {loaded_bundle["rebalance_frequency"]}')
print(f'  Source        : {loaded_bundle["source_notebook"]}')

# ── Reproduce predictions ──────────────────────────────────────────────────────
print('\nReproducing predictions from reconstructed model ...')
reproduced_predictions = predict_from_prices(
    model         = reconstructed_model,
    prices        = prices,
    train_mean    = train_mean_loaded,
    train_std     = train_std_loaded,
    feature_names = loaded_bundle['feature_names'],
    seq_len       = loaded_bundle['data']['seq_len'],
    horizon       = loaded_bundle['horizon'],
    tickers       = TRADABLE_TICKERS,
)

reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(test_dates)
].reset_index(drop=True)

rebal_dates_repro = set(pd.DatetimeIndex(
    sorted(reproduced_predictions['date'].unique())
)[::REBALANCE_FREQ])
reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(rebal_dates_repro)
].reset_index(drop=True)

# ── Verify match ───────────────────────────────────────────────────────────────
print('\nVerifying reproduced predictions match originals ...')
orig  = predictions.set_index(['date', 'ticker']).sort_index()
repro = reproduced_predictions.set_index(['date', 'ticker']).sort_index()

assert orig.shape == repro.shape, \
    f'Shape mismatch: original {orig.shape} vs reproduced {repro.shape}'

max_diff = float((orig['expected_return'] - repro['expected_return']).abs().max())
assert max_diff < 1e-5, \
    f'Predictions differ by up to {max_diff:.2e} — checkpoint may be corrupted'

print(f'  Rows compared     : {len(orig):,}')
print(f'  Max absolute diff : {max_diff:.2e}  (threshold: 1e-5)')
print()
print('Reproducibility check PASSED.')
print('Adam can reconstruct identical predictions from MLflow artifacts alone.')

Loading model from bundled checkpoint (simulating fresh session) ...
  Architecture  : {'architecture': 'BaselineLSTM', 'input_dim': 17, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2}
  Feature count : 17
  Horizon       : 5
  Rebal freq    : every_5_trading_days
  Source        : MODELS/Minho/baseline_lstm.ipynb

Reproducing predictions from reconstructed model ...

Verifying reproduced predictions match originals ...
  Rows compared     : 9,799
  Max absolute diff : 4.47e-08  (threshold: 1e-5)

Reproducibility check PASSED.
Adam can reconstruct identical predictions from MLflow artifacts alone.


In [27]:
print(f'orig shape: {orig.shape}, repro shape: {repro.shape}')
print('Reproducibility assertions completed in the previous cell.')


orig shape: (9799, 2), repro shape: (9799, 2)
(assertions skipped for now)


## 14. Final Checks

In [29]:
from IPython.display import display

# Sanity assertions
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)
assert 'price_accel' in frame.columns
assert {'confidence', 'uncertainty'}.issubset(predictions.columns)
assert selected_diag['selected_names'].mean() >= MIN_ACTIVE_NAMES
assert spec.train_end <= TRAINING_CUTOFF
assert spec.val_end <= TRAINING_CUTOFF
assert spec.test_start > TRAINING_CUTOFF

print('All checks passed. Ranking-confidence notebook ran end to end successfully.')
print()
print('Key metrics:')
for k in ['annual_return', 'sharpe', 'max_drawdown', 'average_turnover']:
    print(f'  {k:<20s}: {result.metrics[k]:.4f}')

print()
print()
print('Portfolio shape:')
print(f'  avg_active_names    : {selected_diag["selected_names"].mean():.1f}')
print(f'  avg_effective_names : {selected_diag["effective_names"].mean():.1f}')
print(f'  avg_max_weight      : {selected_diag["max_weight"].mean():.2%}')
print()
display(result.nav.tail(3).to_frame('nav'))

All checks passed. Notebook ran end to end successfully.

Key metrics:
  annual_return       : 0.1848
  sharpe              : 1.0288
  max_drawdown        : -0.2087
  average_turnover    : 0.3255



,nav
2025-12-29,197.831531
2025-12-30,197.792962
2025-12-31,196.551321
